# Stage 2 Results Review

Use this notebook after syncing new Stage 2 result files back into the repo.

This notebook compares Stage 1 baselines with the new Stage 2 runs and highlights:

- best end-to-end exact match
- best field exact match
- repair deltas
- worst remaining semantic fields

In [1]:
from pathlib import Path
import json
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'results').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not find project root')


PROJECT_ROOT = find_project_root(Path.cwd())
METRICS_DIR = PROJECT_ROOT / 'results' / 'metrics'
print('project_root =', PROJECT_ROOT)
print('metrics_dir =', METRICS_DIR)

project_root = /home/lyan11/small-llm-structured-posttraining
metrics_dir = /home/lyan11/small-llm-structured-posttraining/results/metrics


In [2]:
REPORTS = {
    'prompt_only': METRICS_DIR / 'qwen25_3b_prompt_only_test_report.json',
    'prompt_only_repair': METRICS_DIR / 'qwen25_3b_prompt_only_test_repaired_report.json',
    'qlora_full': METRICS_DIR / 'qwen25_3b_phase1_qlora_v1_test_report.json',
    'qlora_full_repair': METRICS_DIR / 'qwen25_3b_phase1_qlora_v1_test_repaired_report.json',
    'qlora_reduced': METRICS_DIR / 'qwen25_3b_phase1_qlora_reduced_v1_test_report.json',
    'qlora_reduced_repair': METRICS_DIR / 'qwen25_3b_phase1_qlora_reduced_v1_test_repaired_report.json',
    'qlora_reduced_h200fast': METRICS_DIR / 'qwen25_3b_phase1_qlora_reduced_h200fast_test_report.json',
    'schema_generalization': METRICS_DIR / 'qwen25_3b_schema_generalization_v1_test_report.json',
    'stage2_data_small_600': METRICS_DIR / 'qwen25_3b_stage2_data_small_600_test_report.json',
    'stage2_data_small_600_repair': METRICS_DIR / 'qwen25_3b_stage2_data_small_600_test_repaired_report.json',
    'stage2_curriculum': METRICS_DIR / 'qwen25_3b_stage2_curriculum_sm_then_full_test_report.json',
    'stage2_curriculum_repair': METRICS_DIR / 'qwen25_3b_stage2_curriculum_sm_then_full_test_repaired_report.json',
    'stage2_rank8': METRICS_DIR / 'qwen25_3b_stage2_rank8_full_test_report.json',
    'stage2_rank8_repair': METRICS_DIR / 'qwen25_3b_stage2_rank8_full_test_repaired_report.json',
    'stage2_rank16': METRICS_DIR / 'qwen25_3b_stage2_rank16_full_test_report.json',
    'stage2_rank16_repair': METRICS_DIR / 'qwen25_3b_stage2_rank16_full_test_repaired_report.json',
    'stage2_rank32': METRICS_DIR / 'qwen25_3b_stage2_rank32_full_test_report.json',
    'stage2_rank32_repair': METRICS_DIR / 'qwen25_3b_stage2_rank32_full_test_repaired_report.json',
}


def load_json(path):
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))


loaded = {name: load_json(path) for name, path in REPORTS.items()}
missing = [name for name, report in loaded.items() if report is None]
print('missing_reports =', missing)

missing_reports = ['stage2_curriculum', 'stage2_curriculum_repair', 'stage2_rank8', 'stage2_rank8_repair', 'stage2_rank16', 'stage2_rank16_repair', 'stage2_rank32', 'stage2_rank32_repair']


In [3]:
summary_rows = []
for name, report in loaded.items():
    if report is None:
        continue
    summary = report['summary']
    summary_rows.append(
        {
            'experiment': name,
            'num_samples': summary['num_samples'],
            'valid_json_rate': round(summary['valid_json_rate'], 4),
            'schema_compliance_rate': round(summary['schema_compliance_rate'], 4),
            'field_exact_match': round(summary['field_exact_match'], 4),
            'end_to_end_exact_match': round(summary['end_to_end_exact_match'], 4),
        }
    )

summary_rows = sorted(summary_rows, key=lambda item: (item['end_to_end_exact_match'], item['field_exact_match']), reverse=True)
summary_rows

[{'experiment': 'qlora_reduced_h200fast',
  'num_samples': 254,
  'valid_json_rate': 1.0,
  'schema_compliance_rate': 1.0,
  'field_exact_match': 0.8919,
  'end_to_end_exact_match': 0.4961},
 {'experiment': 'qlora_reduced',
  'num_samples': 254,
  'valid_json_rate': 1.0,
  'schema_compliance_rate': 1.0,
  'field_exact_match': 0.8851,
  'end_to_end_exact_match': 0.4882},
 {'experiment': 'qlora_reduced_repair',
  'num_samples': 254,
  'valid_json_rate': 1.0,
  'schema_compliance_rate': 1.0,
  'field_exact_match': 0.8851,
  'end_to_end_exact_match': 0.4882},
 {'experiment': 'schema_generalization',
  'num_samples': 508,
  'valid_json_rate': 0.998,
  'schema_compliance_rate': 0.998,
  'field_exact_match': 0.8764,
  'end_to_end_exact_match': 0.4646},
 {'experiment': 'stage2_data_small_600',
  'num_samples': 254,
  'valid_json_rate': 1.0,
  'schema_compliance_rate': 1.0,
  'field_exact_match': 0.7645,
  'end_to_end_exact_match': 0.0394},
 {'experiment': 'stage2_data_small_600_repair',
  'num

In [4]:
repair_pairs = [
    ('prompt_only', 'prompt_only_repair'),
    ('qlora_full', 'qlora_full_repair'),
    ('qlora_reduced', 'qlora_reduced_repair'),
    ('stage2_data_small_600', 'stage2_data_small_600_repair'),
    ('stage2_curriculum', 'stage2_curriculum_repair'),
    ('stage2_rank8', 'stage2_rank8_repair'),
    ('stage2_rank16', 'stage2_rank16_repair'),
    ('stage2_rank32', 'stage2_rank32_repair'),
]

repair_deltas = []
for raw_name, repaired_name in repair_pairs:
    raw = loaded.get(raw_name)
    repaired = loaded.get(repaired_name)
    if raw is None or repaired is None:
        continue
    raw_summary = raw['summary']
    repaired_summary = repaired['summary']
    repair_deltas.append(
        {
            'experiment': raw_name,
            'schema_delta': round(repaired_summary['schema_compliance_rate'] - raw_summary['schema_compliance_rate'], 4),
            'field_delta': round(repaired_summary['field_exact_match'] - raw_summary['field_exact_match'], 4),
            'e2e_delta': round(repaired_summary['end_to_end_exact_match'] - raw_summary['end_to_end_exact_match'], 4),
        }
    )

repair_deltas

[{'experiment': 'prompt_only',
  'schema_delta': 0.9567,
  'field_delta': 0.1791,
  'e2e_delta': 0.0},
 {'experiment': 'qlora_full',
  'schema_delta': 0.0,
  'field_delta': 0.0,
  'e2e_delta': 0.0},
 {'experiment': 'qlora_reduced',
  'schema_delta': 0.0,
  'field_delta': 0.0,
  'e2e_delta': 0.0},
 {'experiment': 'stage2_data_small_600',
  'schema_delta': 0.0,
  'field_delta': 0.0,
  'e2e_delta': 0.0}]

In [ ]:
FIELD_FILES = {
    'qlora_reduced': METRICS_DIR / 'qwen25_3b_phase1_qlora_reduced_v1_field_analysis.json',
    'qlora_reduced_h200fast': METRICS_DIR / 'qwen25_3b_phase1_qlora_reduced_h200fast_field_analysis.json',
    'schema_generalization': METRICS_DIR / 'qwen25_3b_schema_generalization_v1_field_analysis.json',
    'stage2_data_small_600': METRICS_DIR / 'qwen25_3b_stage2_data_small_600_field_analysis.json',
    'stage2_curriculum': METRICS_DIR / 'qwen25_3b_stage2_curriculum_sm_then_full_field_analysis.json',
    'stage2_rank8': METRICS_DIR / 'qwen25_3b_stage2_rank8_full_field_analysis.json',
    'stage2_rank16': METRICS_DIR / 'qwen25_3b_stage2_rank16_full_field_analysis.json',
    'stage2_rank32': METRICS_DIR / 'qwen25_3b_stage2_rank32_full_field_analysis.json',
}


loaded_fields = {name: load_json(path) for name, path in FIELD_FILES.items()}
for name, data in loaded_fields.items():
    if data is None:
        continue
    print('===', name, '===')
    for item in data['worst_fields'][:8]:
        print(item)
    print()

In [ ]:
generalization = loaded.get('schema_generalization')
if generalization is not None:
    print(json.dumps(generalization.get('grouped_summary', {}).get('by_schema_seen_status', {}), indent=2))